# Colab SGLang 스모크 테스트

이 노트북은 저장소를 clone하지 않고 Colab에서 SGLang 이미지·텍스트 서빙 구성을 임시로 기동해 실제 OpenAI 호환 요청을 확인합니다. 각 단계의 결과는 런타임이 중단되어도 가능한 범위까지 /content/sglang_smoke/results.json 및 results.md에 기록됩니다.

권장 런타임은 GPU L4입니다. L4는 L40S와 같은 Ada 세대라 이미지 서버 단독 검증에 적합합니다. 총 VRAM이 37 GiB 이상인 A100급 런타임에서는 이미지·텍스트 동시 기동 단계도 실행합니다. 텍스트 모델 다운로드는 약 21 GB, 이미지 모델 다운로드는 약 10.2 GiB가 예상됩니다.

위에서부터 셀을 순서대로 실행하세요. 첫 환경 셀이 nvidia-smi가 보고한 CUDA 13 지원 여부를 확인한 뒤에만 고정 조합을 설치합니다. CUDA 13을 지원하지 않으면 우회 설치하지 않고 이후 GPU 서버 단계를 SKIP으로 남깁니다. 기본 입력은 PIL이 만드는 1024×1024 합성 상품 이미지이며, 선택 업로드 셀에서 관리자의 이미지를 대신 사용할 수 있습니다.

이미지 요청은 SGLang Diffusion의 /v1/images/generations JSON 및 /v1/images/edits multipart image 필드를 사용합니다. 텍스트 요청은 /v1/chat/completions의 image_url data URI와 response_format: {"type": "json_object"}를 사용합니다. 완료되면 결과 폴더를 zip으로 묶고 Colab 다운로드를 시도합니다.
      


In [ ]:
from __future__ import annotations

import atexit
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

SMOKE_DIR = Path(os.environ.get("SGLANG_SMOKE_DIR", "/content/sglang_smoke"))
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_JSON = SMOKE_DIR / "results.json"
RESULTS_MD = SMOKE_DIR / "results.md"

records: list[dict[str, Any]] = []
STOP_REASON: str | None = None
TEXT_SKIP_REASON: str | None = None
ENV_INFO: dict[str, Any] = {}
TOTAL_VRAM_GIB: float | None = None
COMPUTE_CAPABILITY: float | None = None
CUDA_VERSION: float | None = None
INPUT_IMAGE_PATH: Path | None = None
INPUT_IMAGE_MIME_TYPE: str = "image/png"


class StageSkip(Exception):
    def __init__(self, reason: str, metrics: dict[str, Any] | None = None):
        super().__init__(reason)
        self.reason = reason
        self.metrics = metrics or {}


class StageFailure(Exception):
    def __init__(self, message: str, metrics: dict[str, Any] | None = None):
        super().__init__(message)
        self.metrics = metrics or {}


def _now() -> str:
    return datetime.now(timezone.utc).isoformat()


def _json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def _write_results() -> None:
    payload = {
        "노트북": "colab_sglang_smoke_test",
        "기록 시각": _now(),
        "중단 사유": STOP_REASON,
        "단계": _json_safe(records),
    }
    temporary = RESULTS_JSON.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(RESULTS_JSON)

    lines = [
        "# Colab SGLang 스모크 테스트 결과",
        "",
        f"- 마지막 기록 시각(UTC): {payload['기록 시각']}",
        f"- 중단 사유: {STOP_REASON or '없음'}",
        "",
        "| 단계 | 상태 | 시작 시각 | 종료 시각 | 소요 초 | 핵심 수치 | 오류 메시지 앞부분 |",
        "|---|---|---|---|---:|---|---|",
    ]
    for record in records:
        metrics_text = json.dumps(
            record.get("핵심 수치", {}),
            ensure_ascii=False,
            separators=(",", ":"),
        )
        error_text = str(record.get("오류 메시지 앞부분", "")).replace("|", "\\|")
        lines.append(
            "| {stage} | {status} | {started} | {ended} | {elapsed} | {metrics} | {error} |".format(
                stage=record.get("단계", ""),
                status=record.get("상태", ""),
                started=record.get("시작 시각", ""),
                ended=record.get("종료 시각", ""),
                elapsed=record.get("소요 초", ""),
                metrics=metrics_text.replace("|", "\\|"),
                error=error_text,
            )
        )
    RESULTS_MD.write_text("\n".join(lines) + "\n", encoding="utf-8")


def set_stop(reason: str) -> None:
    global STOP_REASON
    if STOP_REASON is None:
        STOP_REASON = reason
    _write_results()


def record_skip(
    name: str,
    reason: str,
    metrics: dict[str, Any] | None = None,
) -> dict[str, Any]:
    started = _now()
    record = {
        "단계": name,
        "상태": "SKIP",
        "시작 시각": started,
        "종료 시각": started,
        "소요 초": 0.0,
        "핵심 수치": _json_safe(metrics or {}),
        "오류 메시지 앞부분": reason[:1000],
    }
    records.append(record)
    _write_results()
    print(f"[SKIP] {name}: {reason}")
    return record


def run_stage(name: str, function: Callable[[], dict[str, Any] | None]) -> dict[str, Any]:
    started_at = _now()
    started_monotonic = time.monotonic()
    record: dict[str, Any] = {
        "단계": name,
        "상태": "RUNNING",
        "시작 시각": started_at,
        "종료 시각": None,
        "소요 초": None,
        "핵심 수치": {},
        "오류 메시지 앞부분": "",
    }
    records.append(record)
    _write_results()
    print(f"[RUNNING] {name}")

    try:
        returned = function() or {}
        record["상태"] = "PASS"
        record["핵심 수치"] = _json_safe(returned)
    except StageSkip as exc:
        record["상태"] = "SKIP"
        record["핵심 수치"] = _json_safe(exc.metrics)
        record["오류 메시지 앞부분"] = exc.reason[:1000]
        print(f"[SKIP] {name}: {exc.reason}")
    except StageFailure as exc:
        record["상태"] = "FAIL"
        record["핵심 수치"] = _json_safe(exc.metrics)
        record["오류 메시지 앞부분"] = str(exc)[:1000]
        print(f"[FAIL] {name}: {exc}")
    except Exception as exc:
        record["상태"] = "FAIL"
        record["오류 메시지 앞부분"] = str(exc)[:1000]
        print(f"[FAIL] {name}: {exc}")
    finally:
        record["종료 시각"] = _now()
        record["소요 초"] = round(time.monotonic() - started_monotonic, 3)
        _write_results()
    return record


def run_or_skip(
    name: str,
    function: Callable[[], dict[str, Any] | None],
    reason: str | None = None,
    metrics: dict[str, Any] | None = None,
) -> dict[str, Any]:
    if reason:
        return record_skip(name, reason, metrics)
    return run_stage(name, function)


_write_results()
atexit.register(_write_results)
print(f"결과 기록 위치: {RESULTS_JSON}")
print(f"마크다운 기록 위치: {RESULTS_MD}")
      


In [ ]:
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path


def _capture_command(command: list[str], timeout: int = 30) -> tuple[int | None, str]:
    try:
        completed = subprocess.run(
            command,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        output = (completed.stdout or "") + (completed.stderr or "")
        return completed.returncode, output.strip()
    except FileNotFoundError as exc:
        return None, f"{command[0]} not found: {exc}"
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def _parse_float(value: str | None) -> float | None:
    if not value:
        return None
    match = re.search(r"[0-9]+(?:\.[0-9]+)?", value)
    return float(match.group(0)) if match else None


def collect_environment() -> dict[str, Any]:
    global ENV_INFO, TOTAL_VRAM_GIB, COMPUTE_CAPABILITY, CUDA_VERSION

    smi_code, smi_text = _capture_command(["nvidia-smi"], timeout=30)
    query_code, query_text = _capture_command(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version,compute_cap",
            "--format=csv,noheader,nounits",
        ],
        timeout=30,
    )

    gpu_rows: list[dict[str, Any]] = []
    if query_code == 0:
        for raw_line in query_text.splitlines():
            columns = [column.strip() for column in raw_line.split(",")]
            if len(columns) < 4:
                continue
            memory_mib = _parse_float(columns[1])
            gpu_rows.append(
                {
                    "name": columns[0],
                    "memory_total_mib": memory_mib,
                    "memory_total_gib": round(memory_mib / 1024, 3)
                    if memory_mib is not None
                    else None,
                    "driver_version": columns[2],
                    "compute_capability": _parse_float(columns[3]),
                }
            )

    first_gpu = gpu_rows[0] if gpu_rows else {}
    TOTAL_VRAM_GIB = first_gpu.get("memory_total_gib")
    COMPUTE_CAPABILITY = first_gpu.get("compute_capability")
    cuda_match = re.search(r"CUDA Version:\s*([0-9.]+)", smi_text)
    CUDA_VERSION = _parse_float(cuda_match.group(1) if cuda_match else None)

    disk_root = Path("/content") if Path("/content").exists() else Path.cwd()
    disk = shutil.disk_usage(disk_root)
    try:
        total_ram_gib = (
            os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024**3)
        )
    except (AttributeError, ValueError, OSError):
        total_ram_gib = None

    ENV_INFO = {
        "gpu_count": len(gpu_rows),
        "gpus": gpu_rows,
        "gpu_name": first_gpu.get("name"),
        "total_vram_gib": TOTAL_VRAM_GIB,
        "driver_version": first_gpu.get("driver_version"),
        "nvidia_smi_cuda_version": CUDA_VERSION,
        "compute_capability": COMPUTE_CAPABILITY,
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "system_ram_gib": round(total_ram_gib, 3)
        if total_ram_gib is not None
        else None,
        "disk_root": str(disk_root),
        "disk_free_gib": round(disk.free / (1024**3), 3),
        "nvidia_smi_exit_code": smi_code,
        "nvidia_smi_query_exit_code": query_code,
        "nvidia_smi_excerpt": smi_text[:4000],
    }
    print(json.dumps(ENV_INFO, ensure_ascii=False, indent=2))
    return ENV_INFO


run_stage("환경 기록", collect_environment)
      


In [ ]:
def decide_runtime() -> dict[str, Any]:
    global STOP_REASON, TEXT_SKIP_REASON

    cuda13_supported = CUDA_VERSION is not None and CUDA_VERSION >= 13.0
    compute_supported = (
        COMPUTE_CAPABILITY is not None and COMPUTE_CAPABILITY >= 8.0
    )
    total_vram_known = TOTAL_VRAM_GIB is not None
    text_fraction = (
        min(0.90, (19.6 + 3.0) / TOTAL_VRAM_GIB)
        if TOTAL_VRAM_GIB and TOTAL_VRAM_GIB > 0
        else None
    )
    concurrent_fraction = (
        22.35 / TOTAL_VRAM_GIB
        if TOTAL_VRAM_GIB and TOTAL_VRAM_GIB >= 37.0
        else None
    )

    if not cuda13_supported:
        set_stop(
            "nvidia-smi가 CUDA 13 지원을 확인하지 못했습니다. "
            "운영 조합과 다른 우회 설치를 하지 않고 이후 단계를 중단합니다."
        )
    elif not compute_supported:
        set_stop(
            f"compute capability {COMPUTE_CAPABILITY!r}가 8.0 미만이거나 확인되지 않았습니다. "
            "이후 GPU 단계를 SKIP합니다."
        )
    elif not total_vram_known:
        set_stop("총 VRAM을 확인하지 못해 GPU 서버 단계를 실행하지 않습니다.")

    if TOTAL_VRAM_GIB is not None and TOTAL_VRAM_GIB < 20.0:
        TEXT_SKIP_REASON = (
            f"총 VRAM {TOTAL_VRAM_GIB:.3f} GiB가 20 GiB 미만이라 텍스트 서버를 SKIP합니다."
        )

    decision = {
        "cuda13_supported": cuda13_supported,
        "compute_capability_supported": compute_supported,
        "total_vram_known": total_vram_known,
        "total_vram_gib": TOTAL_VRAM_GIB,
        "text_mem_fraction_static": text_fraction,
        "text_fraction_formula": "min(0.90, (19.6+3.0)/총VRAM)",
        "text_fraction_l40s_reference": 0.50,
        "text_l40s_reference_reserved_gib": 22.35,
        "text_fraction_differs_from_l40s_reference": text_fraction != 0.50,
        "concurrent_threshold_gib": 37.0,
        "concurrent_text_mem_fraction_static": concurrent_fraction,
        "concurrent_absolute_text_memory_target_gib": 22.35,
        "stop_reason": STOP_REASON,
        "text_skip_reason": TEXT_SKIP_REASON,
    }
    print(json.dumps(decision, ensure_ascii=False, indent=2))
    return decision


run_stage("판정 규칙", decide_runtime)
      


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
import time


FIXED_PACKAGES = [
    "sglang[diffusion]==0.5.19",
    "bitsandbytes==0.50.2",
    "torch==2.13.0",
    "flashinfer_python[cu13]==0.6.18",
]


def install_fixed_packages() -> dict[str, Any]:
    global STOP_REASON

    if STOP_REASON:
        raise StageSkip(
            STOP_REASON,
            {"pip_attempted": False, "fixed_packages": FIXED_PACKAGES},
        )

    install_started = time.monotonic()
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", *FIXED_PACKAGES],
        check=False,
        capture_output=True,
        text=True,
        timeout=3600,
    )
    install_seconds = round(time.monotonic() - install_started, 3)
    pip_output = (completed.stdout or "") + (completed.stderr or "")
    metrics: dict[str, Any] = {
        "pip_returncode": completed.returncode,
        "pip_seconds": install_seconds,
        "fixed_packages": FIXED_PACKAGES,
        "pip_output_tail": pip_output[-2000:],
    }
    if completed.returncode != 0:
        set_stop(f"지정 패키지 설치 실패: {pip_output[-1000:]}")
        raise StageFailure("고정 패키지 설치가 실패했습니다.", metrics)

    importlib.invalidate_caches()
    versions: dict[str, Any] = {}
    try:
        sglang = importlib.import_module("sglang")
        torch = importlib.import_module("torch")
        bitsandbytes = importlib.import_module("bitsandbytes")
        versions = {
            "sglang": getattr(sglang, "__version__", importlib.metadata.version("sglang")),
            "torch": getattr(torch, "__version__", importlib.metadata.version("torch")),
            "bitsandbytes": getattr(
                bitsandbytes,
                "__version__",
                importlib.metadata.version("bitsandbytes"),
            ),
            "torch_cuda_build": getattr(getattr(torch, "version", None), "cuda", None),
            "torch_cuda_is_available": bool(torch.cuda.is_available()),
        }
    except Exception as exc:
        set_stop(f"설치 후 import 또는 CUDA 확인 실패: {type(exc).__name__}: {exc}")
        metrics["imports_error"] = f"{type(exc).__name__}: {exc}"
        raise StageFailure("sglang/torch/bitsandbytes import 또는 CUDA 확인 실패", metrics) from exc

    metrics["imports"] = versions
    if not versions["torch_cuda_is_available"]:
        set_stop("torch.cuda.is_available()가 False입니다.")
        raise StageFailure("설치 후 CUDA를 사용할 수 없습니다.", metrics)

    return metrics


run_stage("고정 버전 설치", install_fixed_packages)
      


In [ ]:
from pathlib import Path


def create_synthetic_product_image() -> dict[str, Any]:
    global INPUT_IMAGE_PATH, INPUT_IMAGE_MIME_TYPE

    from PIL import Image, ImageDraw

    size = 1024
    image = Image.new("RGB", (size, size), (238, 232, 220))
    draw = ImageDraw.Draw(image)

    draw.ellipse((148, 730, 876, 910), fill=(194, 185, 170))
    draw.rounded_rectangle(
        (210, 190, 814, 790),
        radius=72,
        fill=(38, 55, 79),
        outline=(20, 29, 43),
        width=10,
    )
    draw.rounded_rectangle(
        (292, 270, 732, 610),
        radius=44,
        fill=(231, 145, 66),
        outline=(255, 218, 158),
        width=12,
    )
    draw.ellipse((442, 353, 582, 493), fill=(255, 244, 212))
    draw.line((350, 680, 674, 680), fill=(190, 214, 235), width=18)
    draw.line((385, 724, 639, 724), fill=(190, 214, 235), width=12)
    draw.text((334, 104), "SGLANG SMOKE", fill=(38, 55, 79))

    INPUT_IMAGE_PATH = SMOKE_DIR / "input_product.png"
    INPUT_IMAGE_MIME_TYPE = "image/png"
    image.save(INPUT_IMAGE_PATH, format="PNG")
    return {
        "path": INPUT_IMAGE_PATH,
        "width": image.width,
        "height": image.height,
        "bytes": INPUT_IMAGE_PATH.stat().st_size,
        "source": "PIL 합성 이미지",
    }


run_stage("테스트 입력 이미지", create_synthetic_product_image)
      


## 선택 입력 이미지

기본값은 False라서 업로드 없이 합성 이미지를 사용합니다. 관리자 사진을 쓰려면 아래 셀의 USE_UPLOADED_IMAGE = True로 바꾸고 그 셀을 실행하세요. PNG/JPEG 등 이미지 하나를 선택하면 서버 테스트 입력으로 대체합니다.
      


In [ ]:
USE_UPLOADED_IMAGE = False


def maybe_upload_image() -> dict[str, Any]:
    global INPUT_IMAGE_PATH, INPUT_IMAGE_MIME_TYPE

    if not USE_UPLOADED_IMAGE:
        raise StageSkip("USE_UPLOADED_IMAGE=False이므로 합성 이미지를 유지합니다.")

    try:
        from google.colab import files
    except ImportError as exc:
        raise StageSkip("Colab 환경이 아니어서 업로드를 건너뜁니다.") from exc

    uploaded = files.upload()
    if not uploaded:
        raise StageSkip("업로드한 파일이 없습니다.")

    filename, data = next(iter(uploaded.items()))
    import mimetypes

    suffix = Path(filename).suffix.lower() or ".png"
    INPUT_IMAGE_MIME_TYPE = mimetypes.guess_type(filename)[0] or "application/octet-stream"
    if not INPUT_IMAGE_MIME_TYPE.startswith("image/"):
        raise RuntimeError(f"업로드 파일이 이미지 MIME 타입이 아닙니다: {INPUT_IMAGE_MIME_TYPE}")
    INPUT_IMAGE_PATH = SMOKE_DIR / f"uploaded_input{suffix}"
    INPUT_IMAGE_PATH.write_bytes(data)
    return {
        "path": INPUT_IMAGE_PATH,
        "filename": filename,
        "mime_type": INPUT_IMAGE_MIME_TYPE,
        "bytes": len(data),
        "source": "관리자 업로드",
    }


run_stage("선택 입력 이미지 업로드", maybe_upload_image)
      


In [ ]:
from __future__ import annotations

import base64
import binascii
import json
import os
import re
import signal
import subprocess
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


IMAGE_BASE_URL = "http://127.0.0.1:30001"
TEXT_BASE_URL = "http://127.0.0.1:30000"
IMAGE_MODEL_NAME = "flux-klein"
TEXT_MODEL_NAME = "qwen-text"
IMAGE_MODEL_PATH = "circulus/FLUX.2-klein-9B-bnb-4bit"
TEXT_MODEL_PATH = "cyankiwi/Qwen3.8-27B-AWQ-INT4"
# deploy/docker-compose.yml / .env.example 과 같은 가중치 커밋
IMAGE_MODEL_REVISION = "58c2804f31af12c8888504b96250010c50b55e44"
TEXT_MODEL_REVISION = "6e134bae811fb5adac50ee042ae5f029ac6779aa"
IMAGE_STEPS = 4
ACTIVE_SERVERS: dict[str, tuple[subprocess.Popen[Any], Any]] = {}


# Source: src/local_detail_page_ai/clients.py:318-346 (SglangImageClient.generate).
# SGLang's image handler accepts this JSON at /v1/images/generations.
# Cross-check: sglang/multimodal_gen/runtime/entrypoints/openai/image_api.py
# forwards prompt, model, n, response_format, size, seed, guidance_scale, and num_inference_steps.
def build_image_generation_payload(
    prompt: str,
    *,
    model: str = IMAGE_MODEL_NAME,
    negative_prompt: str = "",
    width: int = 1024,
    height: int = 1024,
    steps: int = IMAGE_STEPS,
    guidance_scale: float | None = None,
    seed: int | None = None,
) -> dict[str, Any]:
    payload: dict[str, Any] = {
        "model": model,
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "n": 1,
        "size": f"{width}x{height}",
        "response_format": "b64_json",
        "num_inference_steps": steps,
    }
    if guidance_scale is not None:
        payload["guidance_scale"] = guidance_scale
    if seed is not None:
        payload["seed"] = seed
    return payload


# Source: src/local_detail_page_ai/clients.py:348-390 (SglangImageClient.edit).
# The multipart field name is image, matching SGLang's OpenAI edit handler.
# Cross-check: sglang/multimodal_gen/runtime/entrypoints/openai/image_api.py
# parses the image form field and the same generation controls.
def build_image_edit_fields(
    prompt: str,
    *,
    model: str = IMAGE_MODEL_NAME,
    width: int = 1024,
    height: int = 1024,
    steps: int = IMAGE_STEPS,
    guidance_scale: float | None = None,
    seed: int | None = None,
    strength: float = 0.30,
) -> dict[str, Any]:
    del strength
    fields: dict[str, Any] = {
        "model": model,
        "prompt": prompt,
        "n": 1,
        "size": f"{width}x{height}",
        "response_format": "b64_json",
        "num_inference_steps": steps,
    }
    if guidance_scale is not None:
        fields["guidance_scale"] = guidance_scale
    if seed is not None:
        fields["seed"] = seed
    return fields


# Source: src/local_detail_page_ai/clients.py:159-202 (MlxServeChatClient.generate_json).
# SGLang's OpenAI-compatible text handler accepts image_url parts in messages.
# Cross-check: sglang/srt/entrypoints/openai/protocol.py
# ChatCompletionRequest supports messages image_url parts and response_format json_object.
def build_text_payload(
    prompt: str,
    *,
    image: bytes,
    mime_type: str = "image/png",
    model: str = TEXT_MODEL_NAME,
    max_tokens: int = 256,
) -> dict[str, Any]:
    image_data_uri = (
        f"data:{mime_type};base64,{base64.b64encode(image).decode('ascii')}"
    )
    return {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_data_uri},
                    },
                ],
            }
        ],
        "stream": False,
        "temperature": 0,
        "max_tokens": max_tokens,
        "response_format": {"type": "json_object"},
    }


def _request_bytes(
    request: Request,
    *,
    timeout: float = 300.0,
) -> bytes:
    try:
        with urlopen(request, timeout=timeout) as response:
            return response.read()
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code}: {detail[:1000]}") from exc
    except URLError as exc:
        raise RuntimeError(f"URL error: {exc}") from exc


def http_get_json(url: str, *, timeout: float = 30.0) -> dict[str, Any]:
    request = Request(url, method="GET")
    return json.loads(_request_bytes(request, timeout=timeout).decode("utf-8"))


def http_json_post(
    url: str,
    payload: dict[str, Any],
    *,
    timeout: float = 900.0,
) -> dict[str, Any]:
    request = Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    return json.loads(_request_bytes(request, timeout=timeout).decode("utf-8"))


def _multipart_body(
    fields: dict[str, Any],
    *,
    file_field: str,
    filename: str,
    content_type: str,
    file_data: bytes,
) -> tuple[bytes, str]:
    boundary = f"----sglang-smoke-{uuid.uuid4().hex}"
    chunks: list[bytes] = []

    for name, value in fields.items():
        chunks.extend(
            [
                f"--{boundary}\r\n".encode("utf-8"),
                f'Content-Disposition: form-data; name="{name}"\r\n\r\n'.encode(
                    "utf-8"
                ),
                str(value).encode("utf-8"),
                b"\r\n",
            ]
        )

    chunks.extend(
        [
            f"--{boundary}\r\n".encode("utf-8"),
            (
                f'Content-Disposition: form-data; name="{file_field}"; '
                f'filename="{filename}"\r\n'
            ).encode("utf-8"),
            f"Content-Type: {content_type}\r\n\r\n".encode("utf-8"),
            file_data,
            b"\r\n",
            f"--{boundary}--\r\n".encode("utf-8"),
        ]
    )
    return b"".join(chunks), f"multipart/form-data; boundary={boundary}"


def http_multipart_post(
    url: str,
    fields: dict[str, Any],
    *,
    file_field: str,
    filename: str,
    content_type: str,
    file_data: bytes,
    timeout: float = 900.0,
) -> dict[str, Any]:
    body, request_content_type = _multipart_body(
        fields,
        file_field=file_field,
        filename=filename,
        content_type=content_type,
        file_data=file_data,
    )
    request = Request(
        url,
        data=body,
        headers={"Content-Type": request_content_type},
        method="POST",
    )
    return json.loads(_request_bytes(request, timeout=timeout).decode("utf-8"))


def image_dimensions(image_bytes: bytes) -> list[int]:
    from io import BytesIO
    from PIL import Image

    with Image.open(BytesIO(image_bytes)) as image:
        return [image.width, image.height]


def image_bytes_from_response(response: dict[str, Any]) -> bytes:
    data = response.get("data")
    item = data[0] if isinstance(data, list) and data else None
    encoded = item.get("b64_json") if isinstance(item, dict) else None
    if isinstance(encoded, str) and encoded:
        try:
            return base64.b64decode(encoded, validate=True)
        except (ValueError, binascii.Error) as exc:
            raise RuntimeError("응답 b64_json이 유효한 base64가 아닙니다.") from exc

    image_url = item.get("url") if isinstance(item, dict) else None
    if isinstance(image_url, str) and image_url:
        return _request_bytes(Request(image_url, method="GET"), timeout=300.0)

    raise RuntimeError("이미지 응답 data[0]에 b64_json 또는 url이 없습니다.")


def message_content_text(response: dict[str, Any]) -> str:
    choices = response.get("choices")
    choice = choices[0] if isinstance(choices, list) and choices else None
    message = choice.get("message") if isinstance(choice, dict) else None
    content = message.get("content") if isinstance(message, dict) else None
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        text_parts = [
            part.get("text", "")
            for part in content
            if isinstance(part, dict) and isinstance(part.get("text"), str)
        ]
        return "".join(text_parts).strip()
    raise RuntimeError("텍스트 응답 choices[0].message.content가 없습니다.")


def parsed_text_json(response: dict[str, Any]) -> tuple[str, dict[str, Any]]:
    raw_text = message_content_text(response)
    candidate = raw_text.strip()
    fence = chr(96) * 3
    if candidate.startswith(fence):
        candidate = re.sub(
            r"^" + re.escape(fence) + r"(?:json)?\s*",
            "",
            candidate,
        )
        candidate = re.sub(re.escape(fence) + r"\s*$", "", candidate)
    parsed = json.loads(candidate)
    if not isinstance(parsed, dict):
        raise RuntimeError("텍스트 응답 JSON이 객체가 아닙니다.")
    return raw_text, parsed


def build_image_command() -> list[str]:
    return [
        "sglang",
        "serve",
        "--model-path",
        IMAGE_MODEL_PATH,
        "--revision",
        IMAGE_MODEL_REVISION,
        "--served-model-name",
        IMAGE_MODEL_NAME,
        "--host",
        "127.0.0.1",
        "--port",
        "30001",
        "--num-gpus",
        "1",
        "--dit-cpu-offload",
        "false",
        "--text-encoder-cpu-offload",
        "false",
    ]


def build_text_command(mem_fraction: float) -> list[str]:
    return [
        "python3",
        "-m",
        "sglang.launch_server",
        "--model-path",
        TEXT_MODEL_PATH,
        "--revision",
        TEXT_MODEL_REVISION,
        "--served-model-name",
        TEXT_MODEL_NAME,
        "--host",
        "127.0.0.1",
        "--port",
        "30000",
        "--mem-fraction-static",
        f"{mem_fraction:.6f}",
        "--context-length",
        "8192",
        "--trust-remote-code",
    ]


def start_server(name: str, command: list[str], log_path: Path) -> subprocess.Popen[Any]:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_handle = log_path.open("w", encoding="utf-8", buffering=1)
    try:
        process = subprocess.Popen(
            command,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            start_new_session=True,
        )
    except Exception:
        log_handle.close()
        raise
    ACTIVE_SERVERS[name] = (process, log_handle)
    return process


def stop_server(name: str, timeout: int = 45) -> dict[str, Any]:
    entry = ACTIVE_SERVERS.pop(name, None)
    if entry is None:
        return {"name": name, "was_running": False}

    process, log_handle = entry
    termination_error = None
    try:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=timeout)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait(timeout=15)
        returncode = process.returncode
    except Exception as exc:
        termination_error = f"{type(exc).__name__}: {exc}"
        try:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=15)
        except Exception:
            pass
        returncode = process.poll()
    finally:
        log_handle.close()

    result = {
        "name": name,
        "was_running": True,
        "returncode": returncode,
    }
    if termination_error:
        result["termination_error"] = termination_error
    return result


def stop_all_servers() -> list[dict[str, Any]]:
    return [stop_server(name) for name in list(ACTIVE_SERVERS)]


def wait_for_health(
    url: str,
    *,
    process: subprocess.Popen[Any] | None = None,
    timeout: int = 1800,
    interval: int = 5,
) -> dict[str, Any]:
    started = time.monotonic()
    last_error = ""
    while time.monotonic() - started < timeout:
        if process is not None and process.poll() is not None:
            raise RuntimeError(
                f"서버 프로세스가 health check 전에 종료되었습니다 "
                f"(returncode={process.returncode}, last_error={last_error[:500]})"
            )
        try:
            response = http_get_json(url, timeout=20)
            return {
                "healthy": True,
                "elapsed_seconds": round(time.monotonic() - started, 3),
                "response": response,
            }
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            time.sleep(min(interval, max(0.1, timeout - (time.monotonic() - started))))
    raise TimeoutError(
        f"health check timeout after {timeout}s: {last_error[:1000]}"
    )


def gpu_memory_used_gib() -> float | None:
    code, output = _capture_command(
        [
            "nvidia-smi",
            "--query-gpu=memory.used",
            "--format=csv,noheader,nounits",
        ],
        timeout=10,
    )
    if code != 0:
        return None
    first = output.splitlines()[0].strip() if output.splitlines() else ""
    value = _parse_float(first)
    return round(value / 1024, 3) if value is not None else None


class GpuMemoryMonitor:
    def __init__(self, interval: float = 0.5):
        self.interval = interval
        self.values: list[float] = []
        self.errors: list[str] = []
        self._stop_event = threading.Event()
        self._thread: threading.Thread | None = None

    def _poll(self) -> None:
        while not self._stop_event.is_set():
            value = gpu_memory_used_gib()
            if value is None:
                self.errors.append("nvidia-smi memory.used 조회 실패")
            else:
                self.values.append(value)
            self._stop_event.wait(self.interval)

    def start(self) -> None:
        self._thread = threading.Thread(
            target=self._poll,
            name="sglang-gpu-memory-monitor",
            daemon=True,
        )
        self._thread.start()

    def stop(self) -> dict[str, Any]:
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=10)
        return self.snapshot()

    def snapshot(self) -> dict[str, Any]:
        return {
            "samples": len(self.values),
            "baseline_or_first_gib": self.values[0] if self.values else None,
            "peak_gib": max(self.values) if self.values else None,
            "last_gib": self.values[-1] if self.values else None,
            "poll_errors": self.errors[:10],
        }


def wait_for_memory_release(
    baseline_gib: float | None,
    *,
    timeout: int = 90,
) -> dict[str, Any]:
    if baseline_gib is None:
        return {
            "released": None,
            "current_gib": gpu_memory_used_gib(),
            "reason": "기준 메모리를 측정하지 못했습니다.",
        }

    deadline = time.monotonic() + timeout
    current = gpu_memory_used_gib()
    while time.monotonic() < deadline:
        if current is not None and current <= baseline_gib + 0.5:
            return {
                "released": True,
                "baseline_gib": baseline_gib,
                "current_gib": current,
            }
        time.sleep(3)
        current = gpu_memory_used_gib()
    return {
        "released": False,
        "baseline_gib": baseline_gib,
        "current_gib": current,
    }


def log_hints(log_path: Path) -> list[str]:
    if not log_path.exists():
        return []
    pattern = re.compile(r"bitsandbytes|4bit|offload|cpu.?offload", re.IGNORECASE)
    return [
        line[:1000]
        for line in log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        if pattern.search(line)
    ][:20]


print("공통 payload builder와 서버 관리 helper를 준비했습니다.")
print("이미지 모델 ID 분기 없이 served-model-name을 요청 model로 전달합니다.")
      


In [ ]:
def image_server_smoke() -> dict[str, Any]:
    if STOP_REASON:
        raise StageSkip(STOP_REASON)
    if INPUT_IMAGE_PATH is None or not INPUT_IMAGE_PATH.exists():
        raise RuntimeError("입력 이미지가 없습니다.")

    image_metrics: dict[str, Any] = {
        "model_path": IMAGE_MODEL_PATH,
        "served_model_name": IMAGE_MODEL_NAME,
        "base_url": IMAGE_BASE_URL,
        "command": build_image_command(),
        "health_timeout_seconds": 1800,
        "image_steps": IMAGE_STEPS,
        "input_image": INPUT_IMAGE_PATH,
        "input_mime_type": INPUT_IMAGE_MIME_TYPE,
    }
    log_path = SMOKE_DIR / "image_server.log"
    baseline = gpu_memory_used_gib()
    image_metrics["baseline_memory_gib"] = baseline
    process = None
    monitor = GpuMemoryMonitor()
    try:
        process = start_server("image", build_image_command(), log_path)
        image_metrics["pid"] = process.pid
        monitor.start()

        health = wait_for_health(
            f"{IMAGE_BASE_URL}/v1/models",
            process=process,
            timeout=1800,
            interval=5,
        )
        image_metrics["startup_seconds"] = health["elapsed_seconds"]
        image_metrics["models_response"] = health["response"]

        source_bytes = INPUT_IMAGE_PATH.read_bytes()
        generation_prompt = (
            "A clean studio product image of a compact orange geometric device "
            "on a neutral background, centered, soft shadow, high detail."
        )
        generation_started = time.monotonic()
        generation_response = http_json_post(
            f"{IMAGE_BASE_URL}/v1/images/generations",
            build_image_generation_payload(
                generation_prompt,
                model=IMAGE_MODEL_NAME,
                width=1024,
                height=1024,
                steps=4,
                guidance_scale=1.0,
                seed=0,
            ),
            timeout=900,
        )
        generation_bytes = image_bytes_from_response(generation_response)
        generation_path = SMOKE_DIR / "image_generation.png"
        generation_path.write_bytes(generation_bytes)
        image_metrics["generation"] = {
            "seconds": round(time.monotonic() - generation_started, 3),
            "bytes": len(generation_bytes),
            "path": generation_path,
            "response_format": "b64_json",
            "requested_size": "1024x1024",
            "response_dimensions": image_dimensions(generation_bytes),
        }

        edit_started = time.monotonic()
        edit_response = http_multipart_post(
            f"{IMAGE_BASE_URL}/v1/images/edits",
            build_image_edit_fields(
                "Keep the product centered and edit the orange panel to a vivid blue.",
                model=IMAGE_MODEL_NAME,
                width=1024,
                height=1024,
                steps=4,
                guidance_scale=1.0,
                seed=0,
            ),
            file_field="image",
            filename=INPUT_IMAGE_PATH.name,
            content_type=INPUT_IMAGE_MIME_TYPE,
            file_data=source_bytes,
            timeout=900,
        )
        edit_bytes = image_bytes_from_response(edit_response)
        edit_path = SMOKE_DIR / "image_edit.png"
        edit_path.write_bytes(edit_bytes)
        image_metrics["edit"] = {
            "seconds": round(time.monotonic() - edit_started, 3),
            "bytes": len(edit_bytes),
            "path": edit_path,
            "response_format": "b64_json",
            "requested_size": "1024x1024",
            "response_dimensions": image_dimensions(edit_bytes),
            "multipart_file_field": "image",
        }
    except StageFailure:
        raise
    except Exception as exc:
        raise StageFailure(
            f"이미지 서버 스모크 요청 실패: {type(exc).__name__}: {exc}",
            image_metrics,
        ) from exc
    finally:
        image_metrics["gpu_memory"] = monitor.stop()
        try:
            image_metrics["server_stop"] = stop_server("image")
        except Exception as exc:
            image_metrics["server_stop_error"] = f"{type(exc).__name__}: {exc}"
        image_metrics["memory_release"] = wait_for_memory_release(baseline)
        image_metrics["log_path"] = log_path
        image_metrics["log_hints_bitsandbytes_4bit_offload"] = log_hints(log_path)
    return image_metrics


run_or_skip("이미지 서버 단독", image_server_smoke, STOP_REASON)
      


In [ ]:
def text_server_smoke() -> dict[str, Any]:
    if STOP_REASON:
        raise StageSkip(STOP_REASON)
    if TEXT_SKIP_REASON:
        raise StageSkip(TEXT_SKIP_REASON)
    if TOTAL_VRAM_GIB is None:
        raise StageSkip("총 VRAM을 확인하지 못했습니다.")
    if INPUT_IMAGE_PATH is None or not INPUT_IMAGE_PATH.exists():
        raise RuntimeError("입력 이미지가 없습니다.")

    mem_fraction = min(0.90, (19.6 + 3.0) / TOTAL_VRAM_GIB)
    text_metrics: dict[str, Any] = {
        "model_path": TEXT_MODEL_PATH,
        "served_model_name": TEXT_MODEL_NAME,
        "base_url": TEXT_BASE_URL,
        "command": build_text_command(mem_fraction),
        "mem_fraction_static": mem_fraction,
        "mem_fraction_formula": "min(0.90, (19.6+3.0)/총VRAM)",
        "l40s_reference_mem_fraction": 0.50,
        "l40s_reference_reserved_gib": 22.35,
        "input_image": INPUT_IMAGE_PATH,
        "input_mime_type": INPUT_IMAGE_MIME_TYPE,
    }
    log_path = SMOKE_DIR / "text_server.log"
    baseline = gpu_memory_used_gib()
    text_metrics["baseline_memory_gib"] = baseline
    process = None
    monitor = GpuMemoryMonitor()
    try:
        process = start_server("text", build_text_command(mem_fraction), log_path)
        text_metrics["pid"] = process.pid
        monitor.start()
        health = wait_for_health(
            f"{TEXT_BASE_URL}/v1/models",
            process=process,
            timeout=1800,
            interval=5,
        )
        text_metrics["startup_seconds"] = health["elapsed_seconds"]
        text_metrics["models_response"] = health["response"]

        prompt = (
            "이 이미지의 주요 색과 모양을 JSON으로 간단히 설명해 주세요. "
            '{"colors": ["..."], "shapes": ["..."]}'
        )
        request_started = time.monotonic()
        response = http_json_post(
            f"{TEXT_BASE_URL}/v1/chat/completions",
            build_text_payload(
                prompt,
                image=INPUT_IMAGE_PATH.read_bytes(),
                mime_type=INPUT_IMAGE_MIME_TYPE,
                model=TEXT_MODEL_NAME,
                max_tokens=256,
            ),
            timeout=900,
        )
        raw_text, parsed = parsed_text_json(response)
        text_metrics["request"] = {
            "seconds": round(time.monotonic() - request_started, 3),
            "json_parsed": True,
            "json_keys": sorted(parsed.keys()),
            "raw_content_prefix": raw_text[:500],
            "image_part": True,
            "response_format": {"type": "json_object"},
        }
    except StageFailure:
        raise
    except Exception as exc:
        raise StageFailure(
            f"텍스트 서버 스모크 요청 실패: {type(exc).__name__}: {exc}",
            text_metrics,
        ) from exc
    finally:
        text_metrics["gpu_memory"] = monitor.stop()
        try:
            text_metrics["server_stop"] = stop_server("text")
        except Exception as exc:
            text_metrics["server_stop_error"] = f"{type(exc).__name__}: {exc}"
        text_metrics["memory_release"] = wait_for_memory_release(baseline)
        text_metrics["log_path"] = log_path
        text_metrics["log_hints_bitsandbytes_4bit_offload"] = log_hints(log_path)
    return text_metrics


run_or_skip(
    "텍스트 서버 단독",
    text_server_smoke,
    STOP_REASON or TEXT_SKIP_REASON,
    {"mem_fraction_formula": "min(0.90, (19.6+3.0)/총VRAM)"},
)
      


In [ ]:
def concurrent_smoke() -> dict[str, Any]:
    if STOP_REASON:
        raise StageSkip(STOP_REASON)
    if TOTAL_VRAM_GIB is None or TOTAL_VRAM_GIB < 37.0:
        raise StageSkip(
            "총 VRAM이 37 GiB 미만이라 동시 기동을 SKIP합니다.",
            {
                "total_vram_gib": TOTAL_VRAM_GIB,
                "concurrent_threshold_gib": 37.0,
            },
        )
    if INPUT_IMAGE_PATH is None or not INPUT_IMAGE_PATH.exists():
        raise RuntimeError("입력 이미지가 없습니다.")

    concurrent_fraction = 22.35 / TOTAL_VRAM_GIB
    metrics: dict[str, Any] = {
        "image_model_path": IMAGE_MODEL_PATH,
        "text_model_path": TEXT_MODEL_PATH,
        "image_served_model_name": IMAGE_MODEL_NAME,
        "text_served_model_name": TEXT_MODEL_NAME,
        "text_mem_fraction_static": concurrent_fraction,
        "text_absolute_target_gib": 22.35,
        "total_vram_gib": TOTAL_VRAM_GIB,
        "condition": "총 VRAM >= 37 GiB",
        "request_count": 2,
    }
    image_log = SMOKE_DIR / "image_server_concurrent.log"
    text_log = SMOKE_DIR / "text_server_concurrent.log"
    processes: dict[str, subprocess.Popen[Any]] = {}
    monitors: dict[str, GpuMemoryMonitor] = {}
    combined_monitor = GpuMemoryMonitor()
    baseline = gpu_memory_used_gib()
    metrics["baseline_memory_gib"] = baseline
    try:
        processes["image"] = start_server(
            "image-concurrent",
            build_image_command(),
            image_log,
        )
        processes["text"] = start_server(
            "text-concurrent",
            build_text_command(concurrent_fraction),
            text_log,
        )
        monitors["image"] = GpuMemoryMonitor()
        monitors["text"] = GpuMemoryMonitor()
        for monitor in monitors.values():
            monitor.start()
        combined_monitor.start()

        health_started = time.monotonic()
        with ThreadPoolExecutor(max_workers=2) as executor:
            image_health_future = executor.submit(
                wait_for_health,
                f"{IMAGE_BASE_URL}/v1/models",
                process=processes["image"],
                timeout=1800,
                interval=5,
            )
            text_health_future = executor.submit(
                wait_for_health,
                f"{TEXT_BASE_URL}/v1/models",
                process=processes["text"],
                timeout=1800,
                interval=5,
            )
            image_health = image_health_future.result()
            text_health = text_health_future.result()
        metrics["health_seconds"] = round(time.monotonic() - health_started, 3)
        metrics["image_models_response"] = image_health["response"]
        metrics["text_models_response"] = text_health["response"]

        image_prompt = (
            "A clean studio product image of a compact orange geometric device "
            "on a neutral background, centered, soft shadow, high detail."
        )
        text_prompt = (
            "이 이미지의 주요 색과 모양을 JSON으로 간단히 설명해 주세요. "
            '{"colors": ["..."], "shapes": ["..."]}'
        )
        source_bytes = INPUT_IMAGE_PATH.read_bytes()

        def do_image_request() -> dict[str, Any]:
            started = time.monotonic()
            response = http_json_post(
                f"{IMAGE_BASE_URL}/v1/images/generations",
                build_image_generation_payload(
                    image_prompt,
                    model=IMAGE_MODEL_NAME,
                    width=1024,
                    height=1024,
                    steps=4,
                    guidance_scale=1.0,
                    seed=1,
                ),
                timeout=900,
            )
            output = image_bytes_from_response(response)
            output_path = SMOKE_DIR / "image_generation_concurrent.png"
            output_path.write_bytes(output)
            return {
                "seconds": round(time.monotonic() - started, 3),
                "bytes": len(output),
                "path": output_path,
                "requested_size": "1024x1024",
                "response_dimensions": image_dimensions(output),
            }

        def do_text_request() -> dict[str, Any]:
            started = time.monotonic()
            response = http_json_post(
                f"{TEXT_BASE_URL}/v1/chat/completions",
                build_text_payload(
                    text_prompt,
                    image=source_bytes,
                    mime_type=INPUT_IMAGE_MIME_TYPE,
                    model=TEXT_MODEL_NAME,
                    max_tokens=256,
                ),
                timeout=900,
            )
            raw_text, parsed = parsed_text_json(response)
            return {
                "seconds": round(time.monotonic() - started, 3),
                "json_parsed": True,
                "json_keys": sorted(parsed.keys()),
                "raw_content_prefix": raw_text[:500],
            }

        request_started = time.monotonic()
        with ThreadPoolExecutor(max_workers=2) as executor:
            image_future = executor.submit(do_image_request)
            text_future = executor.submit(do_text_request)
            image_result = image_future.result()
            text_result = text_future.result()
        metrics["requests"] = {
            "seconds_wall_clock": round(time.monotonic() - request_started, 3),
            "image": image_result,
            "text": text_result,
            "both_succeeded": True,
        }
    except StageFailure:
        raise
    except Exception as exc:
        metrics["requests"] = {
            "both_succeeded": False,
            "error": f"{type(exc).__name__}: {exc}",
        }
        raise StageFailure(
            f"동시 기동 스모크 실패: {type(exc).__name__}: {exc}",
            metrics,
        ) from exc
    finally:
        for name, monitor in monitors.items():
            metrics.setdefault("gpu_memory_by_server", {})[name] = monitor.stop()
        metrics["combined_gpu_memory"] = combined_monitor.stop()
        for name in ("image-concurrent", "text-concurrent"):
            try:
                metrics.setdefault("server_stop", []).append(stop_server(name))
            except Exception as exc:
                metrics.setdefault("server_stop_errors", []).append(
                    f"{name}: {type(exc).__name__}: {exc}"
                )
        metrics["combined_memory_gib_after_stop"] = wait_for_memory_release(baseline)
        metrics["image_log_hints"] = log_hints(image_log)
        metrics["text_log_hints"] = log_hints(text_log)
    return metrics


run_or_skip("동시 기동", concurrent_smoke, STOP_REASON)
      


In [ ]:
import shutil
import sys
from pathlib import Path


def cleanup_servers() -> dict[str, Any]:
    stopped = stop_all_servers()
    return {
        "stopped_servers": stopped,
        "active_servers_after_cleanup": list(ACTIVE_SERVERS),
    }


cleanup_record = run_stage("정리·다운로드", cleanup_servers)

archive_base = SMOKE_DIR.parent / "sglang_smoke_results"
archive_path: Path | None = None
try:
    # 다운로드 전에 최종 결과 기록을 먼저 갱신해 zip에도 상태가 남게 합니다.
    archive_path = Path(
        shutil.make_archive(
            str(archive_base),
            "zip",
            root_dir=SMOKE_DIR.parent,
            base_dir=SMOKE_DIR.name,
        )
    )
    cleanup_record["핵심 수치"].update(
        {
            "archive_path": str(archive_path),
            "download_requested": False,
        }
    )
    try:
        from google.colab import files
        is_colab = True
    except ImportError:
        files = None
        is_colab = False
        cleanup_record["핵심 수치"]["download_note"] = (
            "Colab이 아니므로 google.colab.files.download를 건너뛰었습니다."
        )
    cleanup_record["핵심 수치"]["download_requested"] = is_colab
    _write_results()
    archive_path = Path(
        shutil.make_archive(
            str(archive_base),
            "zip",
            root_dir=SMOKE_DIR.parent,
            base_dir=SMOKE_DIR.name,
        )
    )
    if is_colab and files is not None:
        try:
            files.download(str(archive_path))
        except Exception as exc:
            cleanup_record["핵심 수치"]["download_error"] = (
                f"{type(exc).__name__}: {exc}"
            )
except Exception as exc:
    cleanup_record["상태"] = "FAIL"
    cleanup_record["오류 메시지 앞부분"] = (
        f"결과 zip 생성 실패: {type(exc).__name__}: {exc}"[:1000]
    )
    cleanup_record["핵심 수치"]["archive_error"] = (
        f"{type(exc).__name__}: {exc}"
    )
_write_results()

print(f"최종 결과 JSON: {RESULTS_JSON}")
print(f"최종 결과 Markdown: {RESULTS_MD}")
print(f"결과 압축 파일: {archive_path}")
for record in records:
    print(f"{record['상태']:7} {record['단계']}")
      
